# Data acquisition - EPL 2025/26

This is the Premier League match data from football-data.co.uk (https://www.football-data.co.uk/englandm.php). Column names are explained in https://www.football-data.co.uk/notes.txt. E0 is their code for the Premier League.

For now I just want to load it and see what is in it.

In [1]:
import polars as pl

epl = pl.read_csv("data/E0.csv")
epl.shape

(380, 132)

There are 132 columns but most of them are betting odds (B365H, AvgH, MaxH...). The match stats are in the first 24 or so.

In [2]:
epl.columns[:24]

['Div',
 'Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR']

In [3]:
epl.head(3)

Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,BFDH,BFDD,BFDA,BMGMH,BMGMD,BMGMA,BVH,BVD,BVA,BWH,…,CLCA,LBCH,LBCD,LBCA,PSCH,PSCD,PSCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365C>2.5,B365C<2.5,PC>2.5,PC<2.5,MaxC>2.5,MaxC<2.5,AvgC>2.5,AvgC<2.5,BFEC>2.5,BFEC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
str,str,str,str,str,i64,i64,str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""E0""","""15/08/2025""","""20:00""","""Liverpool""","""Bournemouth""",4,2,"""H""",1,0,"""H""","""A Taylor""",19,10,10,3,7,10,6,7,1,2,0,0,1.3,6.0,8.5,1.3,6.0,9.5,1.29,6.5,9.0,1.3,6.0,8.5,1.32,…,8.0,1.3,5.75,8.0,1.29,6.55,9.75,1.31,6.6,9.5,1.29,6.02,8.68,1.32,6.8,10.0,1.36,3.2,1.41,2.95,1.4,3.2,1.36,3.05,1.41,3.4,-1.75,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
"""E0""","""16/08/2025""","""12:30""","""Aston Villa""","""Newcastle""",0,0,"""D""",0,0,"""D""","""C Pawson""",3,16,3,3,13,11,3,6,1,1,1,0,2.25,3.5,2.9,2.25,3.75,3.1,2.38,3.55,2.95,2.3,3.5,3.0,2.3,…,2.87,2.37,3.5,2.87,2.32,3.63,3.07,2.45,3.5,3.1,2.38,3.41,2.93,2.48,3.55,3.1,1.83,2.03,1.66,2.31,1.83,2.18,1.75,2.04,1.87,2.12,-0.25,2.05,1.8,2.02,1.89,2.06,1.8,1.95,1.74,2.14,1.86
"""E0""","""16/08/2025""","""15:00""","""Brighton""","""Fulham""",1,1,"""D""",0,0,"""D""","""S Barrott""",10,7,4,2,16,15,4,3,3,3,0,0,1.91,3.6,4.0,1.95,3.75,4.0,1.92,3.65,4.0,1.91,3.6,3.9,1.9,…,4.2,1.85,3.6,4.0,1.92,3.66,4.27,1.87,3.75,4.4,1.83,3.64,4.26,1.9,3.8,4.6,1.88,1.98,1.91,2.0,1.91,2.04,1.84,1.93,1.93,2.06,-0.5,1.83,2.03,1.93,2.0,1.84,2.03,1.8,1.96,1.91,2.08


## Date

Date gets read in as a string (dd/mm/yyyy), so I converted it to a date first.

In [4]:
epl = epl.with_columns(pl.col("Date").str.to_date("%d/%m/%Y"))

epl.select(
    first_match=pl.col("Date").min(),
    last_match=pl.col("Date").max(),
    teams=pl.col("HomeTeam").n_unique(),
    referees=pl.col("Referee").n_unique(),
)

first_match,last_match,teams,referees
date,date,u32,u32
2025-08-15,2026-05-24,20,23


## Columns I'm keeping

| column | meaning | kind of data |
|---|---|---|
| Date | match date | quantitative (date) |
| HomeTeam, AwayTeam | teams | nominal |
| Referee | referee | nominal |
| FTHG, FTAG | full time goals (home, away) | quantitative |
| FTR | result: H, D, A | nominal |
| HS, AS | shots | quantitative |
| HST, AST | shots on target | quantitative |
| HF, AF | fouls | quantitative |
| HC, AC | corners | quantitative |
| HY, AY | yellow cards | quantitative |
| HR, AR | red cards | quantitative |

In [5]:
key_cols = ["Date", "HomeTeam", "AwayTeam", "Referee", "FTHG", "FTAG", "FTR",
            "HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC",
            "HY", "AY", "HR", "AR"]

matches = epl.select(key_cols)
matches.schema

Schema([('Date', Date),
        ('HomeTeam', String),
        ('AwayTeam', String),
        ('Referee', String),
        ('FTHG', Int64),
        ('FTAG', Int64),
        ('FTR', String),
        ('HS', Int64),
        ('AS', Int64),
        ('HST', Int64),
        ('AST', Int64),
        ('HF', Int64),
        ('AF', Int64),
        ('HC', Int64),
        ('AC', Int64),
        ('HY', Int64),
        ('AY', Int64),
        ('HR', Int64),
        ('AR', Int64)])

In [6]:
matches.null_count()

Date,HomeTeam,AwayTeam,Referee,FTHG,FTAG,FTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
matches.describe()

statistic,Date,HomeTeam,AwayTeam,Referee,FTHG,FTAG,FTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""380""","""380""","""380""","""380""",380.0,380.0,"""380""",380.0,380.0,380.0,380.0,380.0,380.0,380.0,380.0,380.0,380.0,380.0,380.0
"""null_count""","""0""","""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2026-01-04 07:15:47.368421""",null,null,null,1.526316,1.223684,null,13.839474,11.157895,4.502632,3.871053,10.652632,10.989474,5.394737,4.602632,1.665789,2.081579,0.05,0.052632
"""std""",null,null,null,null,1.169914,1.084779,null,4.949137,4.573217,2.244311,2.112085,3.215295,3.397173,2.729101,2.748469,1.229105,1.318101,0.241204,0.223591
"""min""","""2025-08-15""","""Arsenal""","""Arsenal""","""A Kitchen""",0.0,0.0,"""A""",1.0,3.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""","""2025-11-01""",null,null,null,1.0,0.0,null,11.0,8.0,3.0,2.0,8.0,9.0,3.0,3.0,1.0,1.0,0.0,0.0
"""50%""","""2026-01-03""",null,null,null,1.0,1.0,null,14.0,11.0,4.0,4.0,10.0,11.0,5.0,4.0,2.0,2.0,0.0,0.0
"""75%""","""2026-03-03""",null,null,null,2.0,2.0,null,16.0,14.0,6.0,5.0,13.0,13.0,7.0,6.0,2.0,3.0,0.0,0.0
"""max""","""2026-05-24""","""Wolves""","""Wolves""","""T Robinson""",5.0,5.0,"""H""",35.0,30.0,11.0,11.0,20.0,21.0,14.0,15.0,5.0,6.0,2.0,1.0


No nulls in these columns and the min/max values look normal.

## What is one row?

20 teams playing everyone home and away = 20 * 19 = 380 games, so each team should have 19 home games.

In [8]:
print("unique (date, home, away):", matches.unique(subset=["Date", "HomeTeam", "AwayTeam"]).height)
print("unique (home, away) pairs:", matches.unique(subset=["HomeTeam", "AwayTeam"]).height)

matches.group_by("HomeTeam").len().sort("HomeTeam")

unique (date, home, away): 380
unique (home, away) pairs: 380


HomeTeam,len
str,u32
"""Arsenal""",19
"""Aston Villa""",19
"""Bournemouth""",19
"""Brentford""",19
"""Brighton""",19
…,…
"""Nott'm Forest""",19
"""Sunderland""",19
"""Tottenham""",19


So one row is one match. Home and away stats are both in the same row (H columns and A columns), so it's kind of a wide format.

## Quick look at home vs away

How often each result happens:

In [9]:
(matches.group_by("FTR")
    .len()
    .with_columns(share=(pl.col("len") / pl.col("len").sum()).round(3))
    .sort("len", descending=True))

FTR,len,share
str,u32,f64
"""H""",162,0.426
"""A""",114,0.3
"""D""",104,0.274


Easier to compare if I make it long (one row per team per match, like the wide vs long example from week 2) and then group by is_home.

In [10]:
home = matches.select(
    "Date",
    team=pl.col("HomeTeam"), opponent=pl.col("AwayTeam"),
    goals=pl.col("FTHG"), goals_against=pl.col("FTAG"),
    shots=pl.col("HS"), shots_on_target=pl.col("HST"),
    fouls=pl.col("HF"), corners=pl.col("HC"),
    yellows=pl.col("HY"), reds=pl.col("HR"),
    is_home=pl.lit(True),
)
away = matches.select(
    "Date",
    team=pl.col("AwayTeam"), opponent=pl.col("HomeTeam"),
    goals=pl.col("FTAG"), goals_against=pl.col("FTHG"),
    shots=pl.col("AS"), shots_on_target=pl.col("AST"),
    fouls=pl.col("AF"), corners=pl.col("AC"),
    yellows=pl.col("AY"), reds=pl.col("AR"),
    is_home=pl.lit(False),
)

team_games = pl.concat([home, away]).with_columns(
    result=pl.when(pl.col("goals") > pl.col("goals_against")).then(pl.lit("W"))
             .when(pl.col("goals") == pl.col("goals_against")).then(pl.lit("D"))
             .otherwise(pl.lit("L"))
)
team_games.shape

(760, 13)

In [11]:
(team_games.group_by("is_home")
    .agg(
        win_rate=(pl.col("result") == "W").mean(),
        goals=pl.col("goals").mean(),
        shots=pl.col("shots").mean(),
        shots_on_target=pl.col("shots_on_target").mean(),
        corners=pl.col("corners").mean(),
        fouls=pl.col("fouls").mean(),
        yellows=pl.col("yellows").mean(),
        reds=pl.col("reds").mean(),
    )
    .with_columns(pl.exclude("is_home").round(3))
    .sort("is_home", descending=True))

is_home,win_rate,goals,shots,shots_on_target,corners,fouls,yellows,reds
bool,f64,f64,f64,f64,f64,f64,f64,f64
true,0.426,1.526,13.839,4.503,5.395,10.653,1.666,0.05
false,0.3,1.224,11.158,3.871,4.603,10.989,2.082,0.053


Home teams won more, scored more, and had more shots. They also got fewer yellow cards even though the average number of fouls was about the same.

This is only one season, so it's just descriptive for now, not a test of anything.

## Summary

- 380 rows, 132 columns (kept 19)
- one row = one match
- 2025-08-15 to 2026-05-24, England, 20 teams, 23 referees
- teams, referee, result are strings; stats are ints; Date converted to a date
- no missing values in the columns above
- no possession, xG or attendance in this file

# Historical EPL Team Data

The dataset contains English Premier League team performance data. Each row represents one team's performance during one Premier League season

This dataset is useful for our research question because it contains separate home and away performance statistics, allowing us to compare team performance at home vs away and see the home-field advantage.

In [2]:
import polars as pl

epl_history = pl.read_csv("data/epl_data.csv")
epl_history.shape

(666, 18)

The dataset contains 666 rows and 18 columns. Each row represents one Premier League team's performance during one season

In [3]:
epl_history.columns

['season',
 'team_name',
 'overall_ranking',
 'home_matches_ranking',
 'away_matches_ranking',
 'matches_played',
 'matches_win',
 'matches_draw',
 'matches_lost',
 'points',
 'home_matches_win',
 'home_matches_draw',
 'home_matches_lost',
 'away_matches_win',
 'away_matches_draw',
 'away_matches_lost',
 'home_matches_pts',
 'average_age']

## Relevant Columns 

The columns most relevant to our research question are:
- 'season' - Premier League season
- 'team_name' - name of the team
- 'home_matches_ranking' - team ranking based on home performance
- 'away_matches_ranking' - away ranking based on away performance
- 'home_matches_win' - number of home matches won
- 'home_matches_draw' - number of home matches drawn
- 'home_matches_lost' - number of home matches lost
- 'away_matches_win' - number of away matches won
- 'away_matches_draw' - number of away matches drawn
- 'away_matches_lost' - number of away matches lost
- 'home_matches_pts' - points earned from home matches

These variables will allow us to compare home and away performance for each team across Premier League seasons

In [4]:
epl_history.select([
    "season",
    "team_name",
    "home_matches_ranking",
    "away_matches_ranking",
    "home_matches_win",
    "home_matches_draw",
    "home_matches_lost",
    "away_matches_win",
    "away_matches_draw",
    "away_matches_lost",
    "home_matches_pts"
]).schema

Schema([('season', Int64),
        ('team_name', String),
        ('home_matches_ranking', Int64),
        ('away_matches_ranking', Int64),
        ('home_matches_win', Int64),
        ('home_matches_draw', Int64),
        ('home_matches_lost', Int64),
        ('away_matches_win', Int64),
        ('away_matches_draw', Int64),
        ('away_matches_lost', Int64),
        ('home_matches_pts', Int64)])

In [6]:
epl_history.select(
    pl.col("season").min().alias("earliest_season"),
    pl.col("season").max().alias("latest_season")
)

earliest_season,latest_season
i64,i64
1992,2024


### Dataset Summary

The historical EPL dataset contains **666 rows and 18 columns**. Each row represents one English Premier League team's performance during one season. 

The dataset covers the **English Premier League in England** from **1992 through 2024**. The variables most relevant to our research question include the season, team name, home and away rankings, home wins, draws, and losses, away wins, draws, and losses, and home points.

The 'team_name' variable is stored as a String, while 'season' and the relevant home and away performance statistics are stored as Int64 values.

This dataset will help answer our question by allowing us to compare teams **home performance against their away performance** across multiple Premier League seasons and determine whether there is evidence of a home-field advantage.


In [7]:
epl_history.head()

season,team_name,overall_ranking,home_matches_ranking,away_matches_ranking,matches_played,matches_win,matches_draw,matches_lost,points,home_matches_win,home_matches_draw,home_matches_lost,away_matches_win,away_matches_draw,away_matches_lost,home_matches_pts,average_age
i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64
1992,"""Manchester United""",1,1,1,42,24,12,6,84,14,5,2,10,7,4,47,25.1
1992,"""Aston Villa""",2,4,2,42,21,11,10,74,13,5,3,8,6,7,44,26.1
1992,"""Norwich City""",3,2,6,42,21,9,12,72,13,6,2,8,3,10,45,25.5
1992,"""Blackburn Rovers""",4,6,3,42,20,11,11,71,13,4,4,7,7,7,43,25.4
1992,"""Queens Park Rangers""",5,8,9,42,17,12,13,63,11,5,5,6,7,8,38,26.8
